# Projet F3 — Prévision de l'inflation
**Contrôle continu — Partie B · Machine Learning appliqué à la finance**
S7 FIN et CI · Module BD & DS · 2026–2027

| | |
|---|---|
| **Nom** | HAKKAK Meriem |
| **Code Apogée** | 25007641 |
| **Groupe** | Fin 2 |
| **Sujet** | F3 — Prévision de l'inflation (régression sur séries mensuelles) |
| **Algorithme principal** | Régression Ridge (régression linéaire régularisée) |
| **Alternatives comparées** | Régression linéaire (MCO) et forêt aléatoire |

## A. Formuler le problème

**Intérêt financier.** L'inflation fixe les taux d'intérêt, la valeur réelle des placements, le coût de la dette et la politique des banques centrales. Anticiper l'inflation du mois prochain aide à décider d'une allocation (obligations indexées, taux fixe ou variable) et à gérer le risque de taux.

**Question.** Peut-on prévoir l'inflation mensuelle américaine du mois *t+1* avec les informations connues au mois *t* (inflation passée, chômage, taux directeur, pétrole), et mieux que des règles naïves ?

**Cible.** `cible` = inflation mensuelle du mois *t+1*, en % : 100 × [ln(IPC<sub>t+1</sub>) − ln(IPC<sub>t</sub>)].

**Tâche.** Régression sur série temporelle mensuelle (horizon : 1 mois).

**Métrique de réussite (fixée avant de voir les résultats).**
- Métrique principale : **RMSE sur le jeu de test** (en points de % par mois).
- Métriques secondaires : **MAE** et **R² hors échantillon** (par rapport à la moyenne d'entraînement).
- **Critère de succès** : le modèle est jugé utile si son RMSE de test est strictement inférieur à celui de la **meilleure référence naïve** (moyenne d'entraînement, persistance ou moyenne mobile 12 mois).

## B. Comprendre les données

In [ ]:
import os, random, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Chemins relatifs : on se place à la racine du dépôt
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
for d in ["data/raw", "figures", "rapport", "docs", "presentation", "video"]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
print("Dossier de travail :", os.getcwd())

In [ ]:
# Séries FRED (Federal Reserve Bank of St. Louis), fréquence mensuelle
SERIES = {
    "CPIAUCSL":   "Indice des prix à la consommation (IPC), tous ménages urbains, désaisonnalisé (1982-84=100) — source BLS",
    "UNRATE":     "Taux de chômage, % de la population active, désaisonnalisé — source BLS",
    "FEDFUNDS":   "Taux des fonds fédéraux effectif, % — source Board of Governors (Fed)",
    "MCOILWTICO": "Prix du pétrole WTI (Cushing), $ par baril, moyenne mensuelle — source EIA",
}
FICHIER_DATE = "data/raw/date_extraction.txt"

def charger_serie(sid):
    chemin = f"data/raw/{sid}.csv"
    if not os.path.exists(chemin):                       # téléchargement une seule fois
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}"
        pd.read_csv(url).to_csv(chemin, index=False)
        with open(FICHIER_DATE, "w") as f:
            f.write(str(datetime.date.today()))
    df = pd.read_csv(chemin)
    df.columns = ["date", sid]
    df["date"] = pd.to_datetime(df["date"])
    df[sid] = pd.to_numeric(df[sid], errors="coerce")
    return df.set_index("date")[sid]

raw = pd.concat([charger_serie(s) for s in SERIES], axis=1).loc["1990-01-01":]
date_extraction = open(FICHIER_DATE).read().strip() if os.path.exists(FICHIER_DATE) else "inconnue"

print("Date d'extraction :", date_extraction)
print("Dimensions brutes :", raw.shape, "| période :", raw.index.min().date(), "→", raw.index.max().date())
raw.tail()

**Fiche des données**

| Élément | Valeur |
|---|---|
| Nom | IPC (CPIAUCSL), chômage (UNRATE), taux directeur (FEDFUNDS), pétrole WTI (MCOILWTICO) |
| Organismes | BLS, Board of Governors of the Federal Reserve System, EIA, diffusés par FRED (Federal Reserve Bank of St. Louis) |
| URL | https://fred.stlouisfed.org/series/CPIAUCSL (et `UNRATE`, `FEDFUNDS`, `MCOILWTICO`) |
| Date d'extraction | affichée ci-dessus (fichier `data/raw/date_extraction.txt`) |
| Licence | Données publiques américaines diffusées par FRED ; conditions d'utilisation : https://fred.stlouisfed.org/legal/ (à vérifier et recopier dans `docs/`) |
| Taille | ≈ 430 lignes × 4 colonnes (voir dimensions ci-dessus) |
| Période / fréquence | janvier 1990 → dernière date disponible, **mensuelle** |
| Unité d'observation | un mois (série agrégée des États-Unis) |

In [ ]:
# Variables construites à partir des séries brutes (information disponible au mois t)
raw = raw.copy()
print("Valeurs manquantes par série (brut) :"); print(raw.isna().sum()); print()
print("Doublons de dates :", int(raw.index.duplicated().sum()))
# Décision : on garde les mois où les 4 séries existent (période commune)
raw = raw.dropna()
print("Période commune retenue :", raw.index.min().date(), "→", raw.index.max().date(), "|", len(raw), "mois")

base = pd.DataFrame(index=raw.index)
base["infl"]          = 100 * np.log(raw["CPIAUCSL"]).diff()      # inflation mensuelle (%)
base["d_chomage"]     = raw["UNRATE"].diff()                       # variation du chômage (pt)
base["d_taux_fed"]    = raw["FEDFUNDS"].diff()                     # variation du taux directeur (pt)
base["petrole_var"]   = 100 * np.log(raw["MCOILWTICO"]).diff()     # variation du pétrole (%)

# Variables explicatives : uniquement le PASSÉ (shift positif) et le mois courant t
X_all = pd.DataFrame(index=base.index)
for k in [0, 1, 2, 3, 11]:
    X_all[f"infl_l{k}"] = base["infl"].shift(k)
X_all["infl_moy3"]       = base["infl"].rolling(3).mean()
X_all["infl_moy12"]      = base["infl"].rolling(12).mean()
X_all["d_chomage"]       = base["d_chomage"]
X_all["d_taux_fed"]      = base["d_taux_fed"]
X_all["petrole_var_l0"]  = base["petrole_var"]
X_all["petrole_var_l1"]  = base["petrole_var"].shift(1)

# Cible : inflation du mois SUIVANT (shift négatif appliqué à la cible uniquement)
y_all = base["infl"].shift(-1).rename("cible")

data = pd.concat([X_all, y_all], axis=1)
n_avant = len(data)
data = data.dropna()          # retire seulement les premières lignes (retards) et le dernier mois (cible inconnue)
print(f"Lignes retirées (retards au début, cible inconnue à la fin) : {n_avant - len(data)}")
print("Jeu de modélisation :", data.shape)
FEATURES = [c for c in data.columns if c != "cible"]

In [ ]:
dico = pd.DataFrame([
    ["infl_l0",       "Explicative", "float", "% par mois",   "Inflation du mois courant t"],
    ["infl_l1",       "Explicative", "float", "% par mois",   "Inflation du mois t-1"],
    ["infl_moy12",    "Explicative", "float", "% par mois",   "Moyenne de l'inflation sur 12 mois (jusqu'à t)"],
    ["d_chomage",     "Explicative", "float", "points de %",  "Variation mensuelle du taux de chômage"],
    ["d_taux_fed",    "Explicative", "float", "points de %",  "Variation mensuelle du taux des fonds fédéraux"],
    ["petrole_var_l0","Explicative", "float", "% par mois",   "Variation mensuelle du prix du WTI"],
    ["cible",         "Cible",       "float", "% par mois",   "Inflation du mois t+1"],
])
dico["Manquants"] = [int(data[v].isna().sum()) for v in dico.iloc[:,0]]
dico.columns = ["Nom", "Rôle", "Type", "Unité", "Description", "Manquants (jeu final)"]
dico

## C. Explorer et préparer

### Découpage chronologique (fait AVANT toute exploration)
- **Entraînement/validation** : jusqu'à novembre 2016 ; la validation se fait par `TimeSeriesSplit` à l'intérieur de cette période (le futur n'est jamais utilisé pour prédire le passé).
- **Test** : de janvier 2017 à la fin des données, **utilisé une seule fois** à la fin.
- **Embargo d'un mois** : décembre 2016 est retiré, car sa cible est l'inflation de janvier 2017 (qui appartient au test).

L'exploration ci-dessous ne porte que sur l'entraînement.

In [ ]:
DATE_TEST = pd.Timestamp("2017-01-01")
train = data[data.index < DATE_TEST - pd.DateOffset(months=1)]   # embargo : on retire déc. 2016
test  = data[data.index >= DATE_TEST]

X_train, y_train = train[FEATURES], train["cible"]
X_test,  y_test  = test[FEATURES],  test["cible"]
print(f"Entraînement : {train.index.min().date()} → {train.index.max().date()} ({len(train)} mois)")
print(f"Test         : {test.index.min().date()} → {test.index.max().date()} ({len(test)} mois)")

In [ ]:
# Valeurs manquantes, doublons, valeurs extrêmes (entraînement)
print("Manquants :", int(train.isna().sum().sum()), "| Doublons de dates :", int(train.index.duplicated().sum()))
q1, q3 = train["cible"].quantile([.25, .75]); iqr = q3 - q1
bornes = (q1 - 3*iqr, q3 + 3*iqr)
extremes = train[(train["cible"] < bornes[0]) | (train["cible"] > bornes[1])]["cible"]
print(f"Valeurs extrêmes de la cible (règle 3×IQR, bornes {bornes[0]:.2f} / {bornes[1]:.2f}) :", len(extremes))
extremes.round(2)

**Décisions de nettoyage.** Aucun doublon et aucune valeur manquante après restriction à la période commune. Les valeurs extrêmes correspondent à des événements économiques réels (chocs pétroliers, crise de 2008). Les supprimer reviendrait à retirer précisément les situations que l'on veut prévoir ; **on les conserve**. L'imputation (médiane) et la standardisation sont placées **dans le Pipeline**, donc ajustées sur l'entraînement uniquement.

In [ ]:
# Figure 1 : inflation mensuelle (entraînement)
fig, ax = plt.subplots()
infl_tr = base.loc[train.index, "infl"]
ax.plot(infl_tr.index, infl_tr, color="#2563eb", lw=1, label="Inflation mensuelle")
ax.plot(infl_tr.index, infl_tr.rolling(12).mean(), color="#ea580c", lw=2, label="Moyenne mobile 12 mois")
ax.axhline(0, color="black", lw=.8)
ax.set_title("Figure 1 — Inflation mensuelle aux États-Unis (période d'entraînement)")
ax.set_xlabel("Date"); ax.set_ylabel("Variation de l'IPC (% par mois)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig1_inflation.png", dpi=150); plt.show()

**Interprétation (figure 1)** *(à relire et adapter avec ton graphique)*. L'inflation mensuelle oscille autour d'environ 0,2 % par mois, avec des fluctuations de court terme importantes. La moyenne mobile 12 mois est beaucoup plus lisse : la tendance est persistante, ce qui rend la persistance et la moyenne mobile des références naïves exigeantes. Les creux marqués (fin 2008) coïncident avec les chutes du prix du pétrole et la crise financière.

In [ ]:
# Figure 2 : distribution de la cible
fig, ax = plt.subplots()
ax.hist(y_train, bins=35, color="#0d9488", edgecolor="white")
ax.axvline(y_train.mean(), color="#ea580c", lw=2, label=f"Moyenne = {y_train.mean():.2f}")
ax.axvline(y_train.median(), color="#7c3aed", lw=2, ls="--", label=f"Médiane = {y_train.median():.2f}")
ax.set_title("Figure 2 — Distribution de l'inflation mensuelle du mois suivant (entraînement)")
ax.set_xlabel("Inflation mensuelle (% par mois)"); ax.set_ylabel("Nombre de mois"); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig2_distribution.png", dpi=150); plt.show()
print("Asymétrie (skewness) :", round(float(y_train.skew()), 2), "| Écart-type :", round(float(y_train.std()), 3))

**Interprétation (figure 2)** *(à adapter)*. La distribution est concentrée autour de 0,2 % avec une queue gauche (mois de déflation). L'écart-type est faible par rapport à la moyenne : prévoir juste la moyenne donne déjà une erreur modeste, donc le gain possible d'un modèle est limité. La queue gauche justifie la conservation des valeurs extrêmes et le choix du RMSE, qui pénalise les grosses erreurs.

In [ ]:
# Figure 3 : corrélation de chaque variable avec la cible (entraînement)
corr = train[FEATURES + ["cible"]].corr()["cible"].drop("cible").sort_values()
fig, ax = plt.subplots(figsize=(10, 5))
couleurs = ["#dc2626" if v < 0 else "#2563eb" for v in corr]
ax.barh(corr.index, corr.values, color=couleurs)
ax.axvline(0, color="black", lw=.8)
ax.set_title("Figure 3 — Corrélation des variables explicatives avec l'inflation du mois suivant")
ax.set_xlabel("Coefficient de corrélation de Pearson"); ax.set_ylabel("Variable")
plt.tight_layout(); plt.savefig("figures/fig3_correlations.png", dpi=150); plt.show()
corr.round(2).to_frame("corrélation")

**Interprétation (figure 3)** *(à adapter)*. Les variables d'inflation récente (moyennes sur 3 et 12 mois, retards) et la variation du pétrole sont les plus liées à la cible ; chômage et taux directeur sont plus faibles à horizon d'un mois. Les variables d'inflation sont corrélées entre elles (multicolinéarité), ce qui motive une régression **régularisée** (Ridge) plutôt que les MCO simples.

## D. L'algorithme retenu : régression Ridge

**Principe et contexte.** La régression Ridge est une régression linéaire à laquelle on ajoute une pénalité sur la taille des coefficients. Ici les variables explicatives sont nombreuses et corrélées (moyennes mobiles et retards de la même série) : les MCO donnent alors des coefficients instables, alors que Ridge les « rétrécit » et réduit la variance de la prévision. Le modèle est rapide, interprétable (un coefficient par variable) et adapté à un petit échantillon mensuel (~310 observations d'entraînement). **Alternatives comparées** : MCO (sans régularisation) et forêt aléatoire (non linéaire, plus flexible mais plus exposée au surapprentissage sur peu de données).

**Équation.** Les variables sont standardisées. Ridge estime

$$\hat\beta^{ridge}=\arg\min_{\beta_0,\beta}\ \sum_{i=1}^{n}\big(y_i-\beta_0-x_i^{\top}\beta\big)^2+\alpha\,\|\beta\|_2^2
\quad\Longrightarrow\quad \hat\beta=(X^{\top}X+\alpha I)^{-1}X^{\top}y$$

et la prévision est $\hat y_{t+1}=\hat\beta_0+x_t^{\top}\hat\beta$.

**Symboles.** $n$ : nombre de mois d'entraînement ; $y_i$ : inflation du mois suivant ; $x_i\in\mathbb{R}^p$ : vecteur des $p$ variables du mois $i$ ; $\beta_0$ : constante (non pénalisée) ; $\beta\in\mathbb{R}^p$ : coefficients **appris** ; $\alpha\ge 0$ : force de pénalité ; $I$ : matrice identité ; $\|\beta\|_2^2=\sum_j\beta_j^2$.

**Deux hyperparamètres.**
1. `alpha` ($\alpha$) : grand → coefficients proches de 0 (biais ↑, variance ↓, tend vers la moyenne) ; petit → proche des MCO. **Réglé** par validation chronologique.
2. `fit_intercept` : si `True`, une constante $\beta_0$ est estimée (ici `True`, car l'inflation moyenne n'est pas nulle) ; valeur fixée, non réglée.

In [ ]:
# Schéma explicatif original (figure 4)
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(12, 4.2)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 4.2)
etapes = [
    ("Données mensuelles\n(IPC, chômage,\ntaux Fed, pétrole)", "#dbeafe", "#2563eb"),
    ("Variables à l'instant t\n(retards, moyennes\nmobiles, variations)", "#dbeafe", "#2563eb"),
    ("Pipeline (train)\nimputation médiane\n+ standardisation", "#ccfbf1", "#0d9488"),
    ("Ridge : estime β\nmin Σ(erreur)² + α‖β‖²", "#ede9fe", "#7c3aed"),
    ("Prévision\ninflation t+1", "#ffedd5", "#ea580c"),
]
x0 = 0.15
for i, (txt, fc, ec) in enumerate(etapes):
    ax.add_patch(FancyBboxPatch((x0 + i*2.35, 1.7), 2.0, 1.5, boxstyle="round,pad=0.05", fc=fc, ec=ec, lw=2))
    ax.text(x0 + i*2.35 + 1.0, 2.45, txt, ha="center", va="center", fontsize=9)
    if i < len(etapes) - 1:
        ax.annotate("", xy=(x0 + (i+1)*2.35, 2.45), xytext=(x0 + i*2.35 + 2.0, 2.45),
                    arrowprops=dict(arrowstyle="->", lw=2))
ax.add_patch(FancyBboxPatch((6.9, 0.2), 4.6, 0.9, boxstyle="round,pad=0.05", fc="#fef9c3", ec="#ca8a04", lw=2))
ax.text(9.2, 0.65, "Choix de α : validation chronologique (TimeSeriesSplit)\nsur l'entraînement uniquement", ha="center", va="center", fontsize=9)
ax.annotate("", xy=(7.5, 1.7), xytext=(8.0, 1.1), arrowprops=dict(arrowstyle="->", lw=1.5, color="#ca8a04"))
ax.text(0.2, 0.65, "Test (2017 →) : utilisé UNE seule fois,\nmodèle figé", fontsize=9, va="center",
        bbox=dict(boxstyle="round", fc="#ffedd5", ec="#ea580c"))
ax.set_title("Figure 4 — Étapes de l'apprentissage et de la prévision (régression Ridge)")
plt.tight_layout(); plt.savefig("figures/fig4_schema_ridge.png", dpi=150); plt.show()

## E. Expérimenter, comparer et conclure

### Références naïves (calculées sans apprentissage)
1. **Moyenne** : on prévoit toujours la moyenne de l'inflation de l'entraînement.
2. **Persistance** : l'inflation de t+1 est prévue égale à celle de t.
3. **Moyenne mobile 12 mois** : l'inflation de t+1 est prévue égale à la moyenne des 12 derniers mois.

### Modèles et réglage (entraînement/validation uniquement)

In [ ]:
def rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))
def mae(y, p):  return float(mean_absolute_error(y, p))
def r2_oos(y, p, ref):   # R² hors échantillon par rapport à une prévision constante "ref" (moyenne d'entraînement)
    return float(1 - np.sum((y - p)**2) / np.sum((y - ref)**2))

moyenne_train = float(y_train.mean())
refs_train = {
    "Réf. moyenne":           np.full(len(y_train), moyenne_train),
    "Réf. persistance":       X_train["infl_l0"].values,
    "Réf. moyenne mobile 12": X_train["infl_moy12"].values,
}
refs_test = {
    "Réf. moyenne":           np.full(len(y_test), moyenne_train),
    "Réf. persistance":       X_test["infl_l0"].values,
    "Réf. moyenne mobile 12": X_test["infl_moy12"].values,
}

tscv = TimeSeriesSplit(n_splits=5)
prep = lambda: [("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]

# Ridge (modèle principal) : réglage de alpha
pipe_ridge = Pipeline(prep() + [("m", Ridge(fit_intercept=True))])
gs_ridge = GridSearchCV(pipe_ridge, {"m__alpha": np.logspace(-2, 4, 25)}, cv=tscv,
                        scoring="neg_root_mean_squared_error").fit(X_train, y_train)

# MCO (alternative 1)
pipe_ols = Pipeline(prep() + [("m", LinearRegression())]).fit(X_train, y_train)
cv_ols = -np.mean([
    -mean_squared_error(y_train.iloc[v], Pipeline(prep()+[("m", LinearRegression())]).fit(X_train.iloc[t], y_train.iloc[t]).predict(X_train.iloc[v]))**0.5
    for t, v in tscv.split(X_train)])

# Forêt aléatoire (alternative 2)
pipe_rf = Pipeline(prep() + [("m", RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1))])
gs_rf = GridSearchCV(pipe_rf, {"m__max_depth": [2, 3, 5, None], "m__min_samples_leaf": [5, 20]}, cv=tscv,
                     scoring="neg_root_mean_squared_error").fit(X_train, y_train)

print("Ridge : meilleur alpha =", round(gs_ridge.best_params_["m__alpha"], 3), "| RMSE CV =", round(-gs_ridge.best_score_, 4))
print("MCO   : RMSE CV =", round(float(cv_ols), 4))
print("Forêt : meilleurs paramètres =", gs_rf.best_params_, "| RMSE CV =", round(-gs_rf.best_score_, 4))

### Évaluation finale (le jeu de test n'est utilisé qu'ici, une seule fois, modèles figés)

In [ ]:
modeles = {"Ridge (principal)": gs_ridge.best_estimator_, "MCO": pipe_ols, "Forêt aléatoire": gs_rf.best_estimator_}
cv_scores = {"Ridge (principal)": -gs_ridge.best_score_, "MCO": cv_ols, "Forêt aléatoire": -gs_rf.best_score_}

lignes = []
for nom, p in refs_test.items():
    lignes.append([nom, rmse(y_train, refs_train[nom]), np.nan, rmse(y_test, p), mae(y_test, p), r2_oos(y_test.values, p, moyenne_train)])
preds_test = {}
for nom, m in modeles.items():
    p = m.predict(X_test); preds_test[nom] = p
    lignes.append([nom, rmse(y_train, m.predict(X_train)), cv_scores[nom], rmse(y_test, p), mae(y_test, p), r2_oos(y_test.values, p, moyenne_train)])

resultats = pd.DataFrame(lignes, columns=["Modèle / référence", "RMSE train", "RMSE CV (train)", "RMSE test", "MAE test", "R² hors éch."]).set_index("Modèle / référence")
resultats.round(4).to_csv("figures/resultats.csv")
resultats.round(4)

In [ ]:
# Figure 5 : prévisions vs réalité sur le test
meilleure_ref = min(refs_test, key=lambda k: rmse(y_test, refs_test[k]))
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(y_test.index, y_test, color="black", lw=1.5, label="Réalité")
ax.plot(y_test.index, preds_test["Ridge (principal)"], color="#7c3aed", lw=1.5, label="Ridge")
ax.plot(y_test.index, refs_test[meilleure_ref], color="#ea580c", lw=1.2, ls="--", label=meilleure_ref)
ax.set_title("Figure 5 — Inflation mensuelle : réalité et prévisions sur la période de test")
ax.set_xlabel("Date d'observation t (la prévision porte sur le mois t+1)"); ax.set_ylabel("Inflation mensuelle (% par mois)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig5_resultats_test.png", dpi=150); plt.show()

In [ ]:
# Analyse des erreurs
err = pd.DataFrame({"réel": y_test, "prévu_ridge": preds_test["Ridge (principal)"]})
err["erreur"] = err["réel"] - err["prévu_ridge"]
print("Plus grosses erreurs absolues (Ridge) :")
display(err.reindex(err["erreur"].abs().sort_values(ascending=False).index).head(6).round(3))

# Erreur par sous-période
for nom, masque in {"2017–2020": err.index < "2021-01-01", "2021 et après (inflation élevée)": err.index >= "2021-01-01"}.items():
    e = err[masque]
    print(f"{nom:35s} | RMSE Ridge = {np.sqrt((e['erreur']**2).mean()):.3f} | n = {len(e)}")

# Coefficients appris (variables standardisées)
coefs = pd.Series(gs_ridge.best_estimator_.named_steps["m"].coef_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(coefs.index, coefs.values, color=["#dc2626" if v < 0 else "#2563eb" for v in coefs])
ax.axvline(0, color="black", lw=.8)
ax.set_title("Figure 6 — Coefficients Ridge (variables standardisées)")
ax.set_xlabel("Coefficient (effet sur l'inflation prévue, en % par écart-type)"); ax.set_ylabel("Variable")
plt.tight_layout(); plt.savefig("figures/fig6_coefficients.png", dpi=150); plt.show()

In [ ]:
# Résumé automatique des résultats
meilleure_ref_rmse = min(rmse(y_test, p) for p in refs_test.values())
rmse_ridge = resultats.loc["Ridge (principal)", "RMSE test"]
gain = 100 * (1 - rmse_ridge / meilleure_ref_rmse)
print(f"Meilleure référence naïve : {meilleure_ref} (RMSE test = {meilleure_ref_rmse:.4f})")
print(f"Ridge                     : RMSE test = {rmse_ridge:.4f}  → écart vs meilleure référence : {gain:+.1f} %")
print("CRITÈRE DE SUCCÈS ATTEINT" if rmse_ridge < meilleure_ref_rmse else "CRITÈRE DE SUCCÈS NON ATTEINT")
ecart_cv_test = resultats.loc["Ridge (principal)", "RMSE test"] - resultats.loc["Ridge (principal)", "RMSE CV (train)"]
print(f"Écart RMSE test − RMSE CV (Ridge) : {ecart_cv_test:+.4f} (un écart important signale un surapprentissage ou un changement de régime)")

### Analyse, surapprentissage et limites *(à rédiger avec tes propres mots après exécution)*
- **Comparaison au naïf** : indique si Ridge bat la meilleure référence et de combien (voir tableau et résumé automatique).
- **Surapprentissage** : compare *RMSE train*, *RMSE CV* et *RMSE test*. Un RMSE train très inférieur au test (surtout pour la forêt) signale du surapprentissage ; Ridge, régularisé, devrait être plus stable.
- **Erreurs** : regarde le tableau des plus grosses erreurs et le RMSE par sous-période. Les erreurs se concentrent probablement sur 2020 (choc COVID) et 2021–2022 (inflation élevée, hors de la plage vue à l'entraînement).
- **Limites** : un seul pays, un seul horizon (1 mois), peu de variables ; le modèle apprend sur 1990–2016, une période de faible inflation, et se généralise mal à un changement de régime ; l'IPC est publié avec retard et peut être révisé ; pas de prévisions d'inflation d'experts ni de variables d'anticipations.
- **Amélioration possible** : ajouter des anticipations d'inflation, tester un horizon à 12 mois, ré-estimer le modèle en fenêtre glissante.

### Conclusion *(réponse explicite à la problématique, à finaliser)*
Réponds en 3–4 phrases : *« Peut-on prévoir l'inflation du mois prochain mieux qu'une règle naïve ? »* → oui/non selon le tableau, avec les chiffres (RMSE test de Ridge vs meilleure référence), puis la limite principale et le résultat à retenir. **Même si Ridge ne bat pas la référence, c'est une réponse valable** : l'inflation mensuelle est très difficile à prévoir et les règles naïves sont déjà solides.